`Lab 4.1: Context management
1. Build a conversation that would exceed roughly 30K tokens; implement a simple strategy — summarize everything older than the last 5 turns into one system-level recap, then drop the raw turns.
2. Measure token count before and after using the response usage field.

In [3]:
#coversation with 30K tokens

import sys
from pathlib import Path
import anthropic
import time
import gradio as gr

sys.path.insert(0, str(Path.cwd().parent))

from common import get_client, MODEL

SYSTEM_PROMPT = "yOU aare an Semiconductor Manufacturing Expert and answer only related to this question."

client = get_client()


In [4]:
"""Lab 4.1, applied to a live Gradio chat: history grows every turn, so
once it passes KEEP_LAST_TURNS we summarize everything older into the
system prompt and only send the recent turns + recap to the API.

Run from the PARENT of ccdv_f_labs/:
    python -m ccdv_f_labs.lab_4_1_gradio_chat
"""

import gradio as gr
import sys
sys.path.insert(0, str(Path.cwd().parent))

from common import get_client, MODEL


SYSTEM_PROMPT = (
    "You are a semiconductor manufacturing expert. Only answer questions "
    "related to semiconductor manufacturing; politely decline anything else."
)
KEEP_LAST_TURNS = 5  # a "turn" = one user + one assistant message

client = get_client()


def count_tokens(messages: list[dict], system: str = "") -> int:
    kwargs = {"model": MODEL, "messages": messages}
    if system:
        kwargs["system"] = system
    return client.messages.count_tokens(**kwargs).input_tokens


def summarize_old_turns(old_messages: list[dict]) -> str:
    transcript = "\n\n".join(f"{m['role'].upper()}: {m['content']}" for m in old_messages)
    resp = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=(
            "Summarize the conversation below into a compact recap. "
            "Preserve concrete facts, decisions, and open questions. "
            "Prose, under 200 words."
        ),
        messages=[{"role": "user", "content": transcript}],
    )
    return resp.content[0].text


def chat(message: str, history: list[dict]) -> str:
    messages = history + [{"role": "user", "content": message}]
    system = SYSTEM_PROMPT

    keep_count = KEEP_LAST_TURNS * 2
    if len(messages) > keep_count:
        old_messages, messages = messages[:-keep_count], messages[-keep_count:]
        recap = summarize_old_turns(old_messages)
        system = f"{SYSTEM_PROMPT}\n\n--- Earlier conversation recap ---\n{recap}"

    tokens = count_tokens(messages, system=system)
    print(f"Sending {len(messages)} messages, {tokens} input tokens")

    resp = client.messages.create(
        model=MODEL, max_tokens=1024, system=system, messages=messages,
    )
    return resp.content[0].text


if __name__ == "__main__":
    gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Sending 1 messages, 54 input tokens
